# Imports

In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
import pydotplus

from sklearn               import metrics as mt
from sklearn               import tree    as tr
from sklearn.tree          import DecisionTreeRegressor
from sklearn.ensemble      import RandomForestRegressor
from IPython.display       import Image
from six                   import StringIO
from sklearn.linear_model  import LinearRegression
from sklearn.linear_model  import Lasso, Ridge, ElasticNet
from sklearn.preprocessing import PolynomialFeatures
from sklearn.cluster       import KMeans
from sklearn.metrics       import silhouette_score
from sklearn.cluster       import AffinityPropagation
from sklearn.preprocessing import StandardScaler

# Carregando os Dados

In [23]:
x_train = pd.read_csv('dados_ensaio/ensaio_clus/dados_agrupa/x_dataset.csv')

# Observando os Dados

In [24]:
x_train.shape

(178, 13)

In [25]:
x_train.describe().T

,count,mean,std,min,25%,50%,75%,max
alcohol,178.0,-8.382808e-16,1.002821,-2.434235,-0.788245,0.061000,0.836129,2.259772
malic_acid,178.0,3.154839e-01,0.220780,0.000000,0.170455,0.222332,0.462945,1.000000
ash,178.0,-8.170742e-16,1.002821,-3.679162,-0.572122,-0.023821,0.698109,3.156325
ash_alcanity,178.0,-3.991813e-17,1.002821,-2.671018,-0.689137,0.001518,0.602088,3.154511
magnesium,178.0,-3.991813e-17,1.002821,-2.088255,-0.824415,-0.122282,0.509638,4.371372
total_phenols,178.0,4.534870e-01,0.215811,0.000000,0.262931,0.474138,0.627586,1.000000
flavanoids,178.0,3.563860e-01,0.210730,0.000000,0.182489,0.378692,0.534810,1.000000
nonflavanoid_phenols,178.0,3.592632e-16,1.002821,-1.868234,-0.740141,-0.176095,0.609541,2.402403
proanthocyanins,178.0,-1.197544e-16,1.002821,-2.069034,-0.597284,-0.062898,0.629175,3.485073
color_intensity,178.0,3.991813e-17,1.002821,-1.634288,-0.795103,-0.159225,0.493956,3.435432


# Ensaios

Vamos medir a performance dos algortimos: K-Mens e Affinity Propagation

A métrica de performanc será: Silhouette Score



# k-Means

In [33]:
#definindo o melhor parametro de k
k_range = np.arange(2, 11, 1)   # começa em 2 — silhouette exige pelo menos 2 clusters

resultados_kmeans = []

for k in k_range:
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)

    model_kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)
    model_kmeans.fit(x_train_scaled)

    labels = model_kmeans.labels_
    wcss = model_kmeans.inertia_ #soma das distâncias ao quadrado de cada ponto até o centro do seu cluster).
    silhouette = silhouette_score(x_train_scaled, labels) #função separada do sklearn, recebe os dados e os rótulos de cluster que o modelo atribuiu.

    resultados_kmeans.append({
        'k': k,
        'WCSS': wcss,
        'Silhouette': silhouette
    })

tabela_kmeans = pd.DataFrame(resultados_kmeans)
tabela_kmeans

,k,WCSS,Silhouette
0,2,1659.007967,0.268313
1,3,1277.928489,0.284859
2,4,1180.737575,0.245721
3,5,1110.365402,0.202610
4,6,1044.469771,0.195992
5,7,995.994619,0.138558
6,8,944.558886,0.158096
7,9,913.197341,0.141743
8,10,864.578095,0.133780


In [34]:
melhor_k = tabela_kmeans.loc[tabela_kmeans['Silhouette'].idxmax()]
melhor_k

k                3.000000
WCSS          1277.928489
Silhouette       0.284859
Name: 1, dtype: float64

In [42]:
#Prevendo k-means com o melhor k
model_kmeans = KMeans(n_clusters=3, random_state=0, n_init=10)
model_kmeans.fit(x_train_scaled)

labels = model_kmeans.labels_
wcss = model_kmeans.inertia_ #soma das distâncias ao quadrado de cada ponto até o centro do seu cluster).
silhouette = silhouette_score(x_train_scaled, labels) #função separada do sklearn, recebe os dados e os rótulos de cluster que o modelo atribuiu.

resultados_kmeans = ({
    'Silhouette': silhouette
})

tabela_kmeans = pd.DataFrame([resultados_kmeans])
tabela_kmeans

,Silhouette
0,0.284859


# Affinity Propagation

In [37]:
#definindo o melhor parametro de preference
preference_range = np.arange(-100, -1, 10)   # ajuste conforme a escala dos seus dados, ver nota abaixo

resultados_ap = []

for p in preference_range:
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)

    model_ap = AffinityPropagation(preference=p, random_state=0)
    model_ap.fit(x_train_scaled)

    #Você não escolhe o número de clusters diretamente — ele emerge do preference. 
    #Por isso guardo n_clusters na tabela, pra você ver quantos clusters cada preference gerou 
    #(útil pra interpretar o resultado, não só o número do Silhouette).
    labels = model_ap.labels_
    n_clusters = len(set(labels))

    # silhouette exige pelo menos 2 clusters
    if n_clusters > 1:
        silhouette = silhouette_score(x_train_scaled, labels)
    else:
        silhouette = np.nan

    resultados_ap.append({
        'preference': p,
        'n_clusters': n_clusters,
        'Silhouette': silhouette
    })

tabela_ap = pd.DataFrame(resultados_ap)
tabela_ap

,preference,n_clusters,Silhouette
0,-100,5,0.160847
1,-90,6,0.152817
2,-80,5,0.180338
3,-70,6,0.152044
4,-60,8,0.104467
5,-50,8,0.115783
6,-40,9,0.123079
7,-30,13,0.118762
8,-20,18,0.124638
9,-10,35,0.130895


In [38]:
melhor_p = tabela_ap.loc[tabela_ap['Silhouette'].idxmax()]
melhor_p

preference   -80.000000
n_clusters     5.000000
Silhouette     0.180338
Name: 2, dtype: float64

In [41]:
#prevendo com o melhor parametro
model_ap = AffinityPropagation(preference=-80, random_state=0)
model_ap.fit(x_train_scaled)

#Você não escolhe o número de clusters diretamente — ele emerge do preference. 
#Por isso guardo n_clusters na tabela, pra você ver quantos clusters cada preference gerou 
#(útil pra interpretar o resultado, não só o número do Silhouette).
labels = model_ap.labels_
n_clusters = len(set(labels))

# silhouette exige pelo menos 2 clusters
if n_clusters > 1:
    silhouette = silhouette_score(x_train_scaled, labels)
else:
    silhouette = np.nan

resultados_ap = ({
    'preference': p,
    'n_clusters': n_clusters,
    'Silhouette': silhouette
})

tabela_ap = pd.DataFrame([resultados_ap])
tabela_ap

,preference,n_clusters,Silhouette
0,-10,5,0.180338
